# Thai Dubbing API บน Google Colab

1. เมนู Runtime > Change runtime type > เลือก **T4 GPU**
2. อัปโหลด `thai-dubbing-api.zip` ไปที่แถบ Files ทางซ้าย
3. ใส่คีย์ใน Secrets (รูปกุญแจทางซ้าย): `HF_TOKEN`, `TYPHOON_API_KEY` และถ้าต้องการ `API_KEY`
4. รันทีละเซลล์จากบนลงล่าง

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!unzip -o -q /content/thai-dubbing-api.zip -d /content/
%cd /content/thai-dubbing-api
!ls

In [ ]:
# ติดตั้งโมเดล ใช้เวลาประมาณ 5-10 นาที
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q -r requirements-ml.txt

In [ ]:
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return ''

env = {
    'DATA_DIR': '/content/data',
    'DEVICE': 'cuda',
    'HF_TOKEN': secret('HF_TOKEN'),
    'TYPHOON_API_KEY': secret('TYPHOON_API_KEY'),
    'API_KEY': secret('API_KEY'),
    'WHISPER_MODEL': 'large-v3',
}
with open('.env', 'w') as f:
    for k, v in env.items():
        f.write(f'{k}={v}\n')
print({k: ('***' if v and ('KEY' in k or 'TOKEN' in k) else v) for k, v in env.items()})

In [ ]:
# เปิดเซิร์ฟเวอร์ไว้เบื้องหลัง
import subprocess, time, requests
server = subprocess.Popen(['python', 'run.py'], stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(requests.get('http://localhost:8000/health').json()); break
    except Exception:
        time.sleep(2)
else:
    print(open('server.log').read()[-3000:])

In [ ]:
# เปิดลิงก์สาธารณะฟรีด้วย Cloudflare Tunnel
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import re
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('ลิงก์ API:', m.group(0))
        print('เอกสาร API:', m.group(0) + '/docs')
        break

In [ ]:
# ดู log ล่าสุดของเซิร์ฟเวอร์ (ใช้ตอนงานล้มเหลว)
print(open('server.log').read()[-4000:])